# Malta Property Market Explorer

## Week 1 - Day 3

### Objective

Create the first dataset from 5 properties.

Create the first dataset from 5 properties.

## Setup

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
#Define URL and headers

import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.propertymarket.com.mt/for-sale/?locations%5B%5D=Birkirkara"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print("Length:", len(response.text))

Status: 200
Length: 109175


In [3]:
#Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>Properties For Sale in Birkirkara - Real Estate</title>


In [4]:
#Extract property links

links = soup.find_all("a")

property_links = []

for link in links:
    href = link.get("href")

    if href and "/view/" in href:
        property_links.append(href)

print("Properties found:", len(property_links))

Properties found: 72


In [5]:
for url in property_links[:5]:
    print(url)

https://www.propertymarket.com.mt/view/3-bedroom-terraced-house-for-sale-birkirkara-4122532301899243817
https://www.propertymarket.com.mt/view/3-bedroom-terraced-house-for-sale-birkirkara-4122532301899243817
https://www.propertymarket.com.mt/view/3-bedroom-terraced-house-for-sale-birkirkara-4122532301899243817
https://www.propertymarket.com.mt/view/3-bedroom-terraced-house-for-sale-birkirkara-4122532301899243817
https://www.propertymarket.com.mt/view/3-bedroom-apartment-for-sale-birkirkara-4122532301899269523


## Remove Duplicate Property Links

In [6]:
print("Links before removing duplicates:")
print(len(property_links))

Links before removing duplicates:
72


In [7]:
property_links = list(set(property_links))

In [8]:
print("Links after removing duplicates:")
print(len(property_links))

Links after removing duplicates:
23


In [9]:
for url in property_links[:5]:
    print(url)

https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446
https://www.propertymarket.com.mt/view/2-bedroom-apartment-for-sale-birkirkara-4122532301899273447
https://www.propertymarket.com.mt/view/1-bedroom-apartment-for-sale-birkirkara-4122532301899273481
https://www.propertymarket.com.mt/view/3-bedroom-penthouse-for-sale-birkirkara-4122532301899264269
https://www.propertymarket.com.mt/view/2-bedroom-terraced-house-for-sale-birkirkara-4122532301899273167/#requestDetails


### Data Quality Check

Property links contained duplicates because the website uses the same URL in multiple page elements.

Duplicates were removed before dataset creation.

In [10]:
property_links = list(dict.fromkeys(property_links))

## Extract Data From One Property

Before processing multiple properties, we test the extraction process on a single listing.

In [11]:
first_property = property_links[0]

print(first_property)

https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446


### Access Property Page

The first step is to request the property page and verify that the website allows access.

In [12]:
property_response = requests.get(
    first_property,
    headers=headers
)

print("Status:", property_response.status_code)
print("Length:", len(property_response.text))

Status: 200
Length: 66189


In [13]:
if property_response.status_code != 200:
    print("Failed to access property")

### Parse Property Page

Convert the HTML response into a BeautifulSoup object for data extraction.

In [14]:
property_soup = BeautifulSoup(
    property_response.text,
    "html.parser"
)

print(property_soup.title.text)

2 Bedroom House For Sale in Birkirkara - 68746


### Find Price Information

Before extracting prices automatically, we inspect the page and locate where the price appears.

In [15]:
price_matches = property_soup.find_all(
    string=lambda text: text and "€" in text
)

for price in price_matches[:10]:
    print(price.strip())

€750,000
€3,379.21 per month*
Amount (€)


### Extract Basic Property Information

Using the title and price information, extract the first structured fields.

In [16]:
title = property_soup.title.text

price = price_matches[0].strip()

print("Title:", title)
print("Price:", price)

Title: 2 Bedroom House For Sale in Birkirkara - 68746
Price: €750,000


### Extract Information From Title

The property title contains useful structured information such as the number of bedrooms, property type, and location.

In [17]:
title_parts = title.split()

print(title_parts)

['2', 'Bedroom', 'House', 'For', 'Sale', 'in', 'Birkirkara', '-', '68746']


### Extract Individual Fields

In [18]:
print("Bedrooms:", title_parts[0])
print("Property Type:", title_parts[2])
print("Location:", title_parts[6])

Bedrooms: 2
Property Type: House
Location: Birkirkara


In [19]:
{
    "bedrooms": 2,
    "property_type": "Apartment",
    "location": "Birkirkara",
    "price": "€335,000"
}

{'bedrooms': 2,
 'property_type': 'Apartment',
 'location': 'Birkirkara',
 'price': '€335,000'}

## Create First Extraction Function

Functions allow us to reuse the same extraction logic for multiple properties.

In [20]:
bedrooms = int(title_parts[0])

property_type = title_parts[2]

location = title_parts[6]

print("Bedrooms:", bedrooms)
print("Property Type:", property_type)
print("Location:", location)
print("Price:", price)

Bedrooms: 2
Property Type: House
Location: Birkirkara
Price: €750,000


### Create a Property Record

Store the extracted property information in a Python dictionary.

In [21]:
property_record = {
    "bedrooms": bedrooms,
    "property_type": property_type,
    "location": location,
    "price": price,
    "url": first_property
}

property_record

{'bedrooms': 2,
 'property_type': 'House',
 'location': 'Birkirkara',
 'price': '€750,000',
 'url': 'https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446'}

### Build a Property Record Function

Create a reusable function that converts extracted values into a structured property record.

In [22]:
def create_property_record(
    bedrooms,
    property_type,
    location,
    price,
    url
):
    
    property_record = {
        "bedrooms": bedrooms,
        "property_type": property_type,
        "location": location,
        "price": price,
        "url": url
    }
    
    return property_record

In [23]:
record = create_property_record(
    bedrooms,
    property_type,
    location,
    price,
    first_property
)

record

{'bedrooms': 2,
 'property_type': 'House',
 'location': 'Birkirkara',
 'price': '€750,000',
 'url': 'https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446'}

### Create a List of Property Records

Collect multiple property records into a list before creating a DataFrame.

In [24]:
property_records = []

print(property_records)

[]


A list is used to store multiple property records before converting them into a pandas DataFrame.

### Add First Property Record

Add the first property record to the list of property records.

In [25]:
property_records.append(record)

print(property_records)

[{'bedrooms': 2, 'property_type': 'House', 'location': 'Birkirkara', 'price': '€750,000', 'url': 'https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446'}]


The append() method adds a property record to the list.

The list will later contain multiple properties and will be converted into a pandas DataFrame.

### Create First DataFrame

Convert the list of property records into a pandas DataFrame.

In [26]:
properties_df = pd.DataFrame(property_records)

properties_df

,bedrooms,property_type,location,price,url
0,2,House,Birkirkara,"€750,000",https://www.propertymarket.com.mt/view/2-bedro...


The DataFrame is the main structure used for data analysis in pandas.

Each dictionary becomes one row in the DataFrame and each key becomes a column.

## Build a Mini Dataset

Create property records from the first five property listings.

In [27]:
for i, url in enumerate(property_links[:5], start=1):
    print(f"Property {i}:")
    print(url)
    print()

Property 1:
https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446

Property 2:
https://www.propertymarket.com.mt/view/2-bedroom-apartment-for-sale-birkirkara-4122532301899273447

Property 3:
https://www.propertymarket.com.mt/view/1-bedroom-apartment-for-sale-birkirkara-4122532301899273481

Property 4:
https://www.propertymarket.com.mt/view/3-bedroom-penthouse-for-sale-birkirkara-4122532301899264269

Property 5:
https://www.propertymarket.com.mt/view/2-bedroom-terraced-house-for-sale-birkirkara-4122532301899273167/#requestDetails



### First Automation Loop

Use a loop to process multiple properties automatically.

In [28]:
for url in property_links[:5]:

    print("Opening property:")
    print(url)
    print("-" * 50)

Opening property:
https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446
--------------------------------------------------
Opening property:
https://www.propertymarket.com.mt/view/2-bedroom-apartment-for-sale-birkirkara-4122532301899273447
--------------------------------------------------
Opening property:
https://www.propertymarket.com.mt/view/1-bedroom-apartment-for-sale-birkirkara-4122532301899273481
--------------------------------------------------
Opening property:
https://www.propertymarket.com.mt/view/3-bedroom-penthouse-for-sale-birkirkara-4122532301899264269
--------------------------------------------------
Opening property:
https://www.propertymarket.com.mt/view/2-bedroom-terraced-house-for-sale-birkirkara-4122532301899273167/#requestDetails
--------------------------------------------------


### Extract Titles From Multiple Properties

Use a loop to access multiple property pages and extract their titles.

In [29]:
for url in property_links[:5]:

    property_response = requests.get(
        url,
        headers=headers
    )

    property_soup = BeautifulSoup(
        property_response.text,
        "html.parser"
    )

    print(property_soup.title.text)
    print("-" * 50)

2 Bedroom House For Sale in Birkirkara - 68746
--------------------------------------------------
2 Bedroom Apartment For Sale in Birkirkara - 68228
--------------------------------------------------
1 Bedroom Apartment For Sale in Birkirkara - SAPT607780
--------------------------------------------------
3 Bedroom Penthouse For Sale in Birkirkara - 2026156
--------------------------------------------------
2 Bedroom Terraced House For Sale in Birkirkara - P000257252
--------------------------------------------------


### Build Property Records Automatically

Extract information from multiple properties and store it in a list.

In [30]:
property_records = []

for url in property_links[:5]:

    property_response = requests.get(
        url,
        headers=headers
    )

    property_soup = BeautifulSoup(
        property_response.text,
        "html.parser"
    )

    title = property_soup.title.text

    print(title)

2 Bedroom House For Sale in Birkirkara - 68746
2 Bedroom Apartment For Sale in Birkirkara - 68228
1 Bedroom Apartment For Sale in Birkirkara - SAPT607780
3 Bedroom Penthouse For Sale in Birkirkara - 2026156
2 Bedroom Terraced House For Sale in Birkirkara - P000257252


In [32]:
property_records

[{'bedrooms': 2,
  'property_type': 'House',
  'location': 'Birkirkara',
  'price': '€750,000',
  'url': 'https://www.propertymarket.com.mt/view/2-bedroom-house-for-sale-birkirkara-4122532301899273446'},
 {'bedrooms': 2,
  'property_type': 'Apartment',
  'location': 'Birkirkara',
  'price': '€260,000',
  'url': 'https://www.propertymarket.com.mt/view/2-bedroom-apartment-for-sale-birkirkara-4122532301899273447'},
 {'bedrooms': 1,
  'property_type': 'Apartment',
  'location': 'Birkirkara',
  'price': '€192,000',
  'url': 'https://www.propertymarket.com.mt/view/1-bedroom-apartment-for-sale-birkirkara-4122532301899273481'},
 {'bedrooms': 3,
  'property_type': 'Penthouse',
  'location': 'Birkirkara',
  'price': '€585,000',
  'url': 'https://www.propertymarket.com.mt/view/3-bedroom-penthouse-for-sale-birkirkara-4122532301899264269'},
 {'bedrooms': 2,
  'property_type': 'Terraced',
  'location': 'in',
  'price': '€450,000',
  'url': 'https://www.propertymarket.com.mt/view/2-bedroom-terraced-h

In [31]:
property_records = []

for url in property_links[:5]:

    property_response = requests.get(
        url,
        headers=headers
    )

    property_soup = BeautifulSoup(
        property_response.text,
        "html.parser"
    )

    title = property_soup.title.text

    title_parts = title.split()

    bedrooms = int(title_parts[0])
    property_type = title_parts[2]
    location = title_parts[6]

    price_matches = property_soup.find_all(
        string=lambda text: text and "€" in text
    )

    price = price_matches[0].strip()

    record = {
        "bedrooms": bedrooms,
        "property_type": property_type,
        "location": location,
        "price": price,
        "url": url
    }

    property_records.append(record)

print("Records collected:", len(property_records))

Records collected: 5


### Create Dataset From Multiple Properties

In [34]:
properties_df = pd.DataFrame(property_records)

properties_df

,bedrooms,property_type,location,price,url
0,2,House,Birkirkara,"€750,000",https://www.propertymarket.com.mt/view/2-bedro...
1,2,Apartment,Birkirkara,"€260,000",https://www.propertymarket.com.mt/view/2-bedro...
2,1,Apartment,Birkirkara,"€192,000",https://www.propertymarket.com.mt/view/1-bedro...
3,3,Penthouse,Birkirkara,"€585,000",https://www.propertymarket.com.mt/view/3-bedro...
4,2,Terraced,in,"€450,000",https://www.propertymarket.com.mt/view/2-bedro...


In [35]:
properties_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   bedrooms       5 non-null      int64 
 1   property_type  5 non-null      object
 2   location       5 non-null      object
 3   price          5 non-null      object
 4   url            5 non-null      object
dtypes: int64(1), object(4)
memory usage: 332.0+ bytes


### Week 1 - Day 3

#### Achievements:

- Scraped property listings from PropertyMarket Malta
- Extracted property URLs
- Built automated extraction loops
- Created structured property records
- Converted records into a pandas DataFrame
- Performed initial data quality checks

## Lessons Learned

- A website may block requests without a User-Agent header.
- Property listings can appear multiple times on the same page.
- Real-world data often contains inconsistencies.
- A DataFrame is created from a list of dictionaries.
- Data quality checks are an important part of any data project.

## Known Issues

One property returned an incorrect location value ("in")
because the extraction logic assumes all titles follow the
same format.

This issue will be fixed during Week 1 - Day 4 as part of
the data cleaning process.